# Premium Subscription Prediction with Ensembles

Random Forest vs. XGBoost on e-commerce behavior data: diagnosing overfitting from the train/test accuracy gap and tuning XGBoost to reduce variance.

_Datasets are not included in this repository._

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

df = pd.read_csv('ecommerce_premium_behavior.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   age                    2500 non-null   int64  
 1   monthly_spend          2500 non-null   float64
 2   app_usage_hours        2500 non-null   float64
 3   loyalty_months         2500 non-null   int64  
 4   random_activity_score  2500 non-null   float64
 5   device_type            2500 non-null   object 
 6   city_tier              2500 non-null   object 
 7   is_premium             2500 non-null   int64  
dtypes: float64(3), int64(3), object(2)
memory usage: 156.4+ KB


In [15]:
df.head(10)

,age,monthly_spend,app_usage_hours,loyalty_months,random_activity_score,device_type,city_tier,is_premium
0,56,1.229152e+06,23.377286,7,97.467190,Web,Tier 2,0
1,46,1.879253e+06,55.554044,52,23.309407,Android,Tier 3,1
2,32,1.764600e+06,52.832794,27,77.314860,Android,Tier 2,1
3,60,6.277300e+05,28.943985,9,8.760670,Android,Tier 2,0
4,25,1.717752e+06,30.509489,22,38.157592,Web,Tier 1,0
5,38,1.006318e+05,28.506671,34,50.668815,Android,Tier 1,0
6,56,1.336068e+05,77.379694,46,26.874695,Android,Tier 1,0
7,36,1.244851e+06,65.538186,49,52.957842,Android,Tier 1,1
8,40,7.931166e+05,83.409301,1,88.261443,iOS,Tier 2,0
9,28,8.342756e+05,97.571830,9,94.443796,Android,Tier 1,0


In [18]:
df.tail(10)

,age,monthly_spend,app_usage_hours,loyalty_months,random_activity_score,device_type,city_tier,is_premium
2490,38,9.092816e+05,42.110882,25,42.732983,iOS,Tier 1,1
2491,51,1.989900e+06,29.071253,22,77.415679,Android,Tier 1,0
2492,21,2.757508e+05,29.103940,28,47.040762,Android,Tier 2,0
2493,42,1.582661e+06,48.746841,39,29.522825,iOS,Tier 1,1
2494,32,1.380033e+06,72.238343,23,73.926515,Android,Tier 1,1
2495,21,1.558248e+06,36.528000,4,79.820213,iOS,Tier 2,0
2496,62,6.843066e+05,61.394291,52,20.284079,Android,Tier 2,0
2497,29,1.034904e+06,38.865962,4,30.516950,Android,Tier 1,0
2498,49,1.613366e+06,31.147796,47,49.477684,Android,Tier 1,1
2499,49,5.418285e+05,3.461092,59,18.652183,Web,Tier 2,0


In [10]:
df.describe()

,age,monthly_spend,app_usage_hours,loyalty_months,random_activity_score,is_premium
count,2500.000000,2.500000e+03,2500.000000,2500.000000,2500.000000,2500.00000
mean,41.129600,1.055475e+06,49.754007,29.344400,50.319344,0.34280
std,13.558464,5.483676e+05,28.556171,16.996886,28.715399,0.47474
min,18.000000,1.003580e+05,1.018466,1.000000,0.024067,0.00000
25%,29.000000,5.788009e+05,25.162054,15.000000,25.551037,0.00000
50%,42.000000,1.073356e+06,49.841033,29.000000,50.064409,0.00000
75%,53.000000,1.533256e+06,73.747962,44.000000,74.805541,1.00000
max,64.000000,1.999456e+06,99.985075,59.000000,99.989267,1.00000


In [12]:
df.isna().sum()

,0
age,0
monthly_spend,0
app_usage_hours,0
loyalty_months,0
random_activity_score,0
device_type,0
city_tier,0
is_premium,0


In [13]:
df.duplicated().sum()

np.int64(0)

In [21]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [23]:
x = df.drop('is_premium', axis=1)
y = df['is_premium']

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42,stratify=y)

In [112]:
onehot = ['device_type', 'city_tier']
CT = ColumnTransformer([('onehot', OneHotEncoder(drop='first'), onehot)], remainder='passthrough')
Pipe = Pipeline([('CT', CT), ('RF', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))])
model = Pipe.fit(X_train, y_train)
accuracy_score(y_train, model.predict(X_train)), accuracy_score(y_test, model.predict(X_test)), accuracy_score(y_train, model.predict(X_train)) - accuracy_score(y_test, model.predict(X_test))

(0.86, 0.838, 0.02200000000000002)

In [113]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
cm = confusion_matrix(y_test, model.predict(X_test))
print(cm)
print(classification_report(y_test, model.predict(X_test)))

[[313  16]
 [ 65 106]]
              precision    recall  f1-score   support

           0       0.83      0.95      0.89       329
           1       0.87      0.62      0.72       171

    accuracy                           0.84       500
   macro avg       0.85      0.79      0.80       500
weighted avg       0.84      0.84      0.83       500



Cukup Bagus

In [114]:
Pipe = Pipeline([('CT', CT), ('XGB', XGBClassifier(n_estimators=200, learning_rate=0.3, max_depth=12))])
model = Pipe.fit(X_train, y_train)
accuracy_score(y_train, model.predict(X_train)), accuracy_score(y_test, model.predict(X_test)), accuracy_score(y_train, model.predict(X_train)) - accuracy_score(y_test, model.predict(X_test))

(1.0, 0.846, 0.15400000000000003)

 Berdasarkan analisis tersebut, model terlihat overfit dikarenakan menghafal training dataset dan melewati pola yang ada. Penyebab ini kemungkinan dari besarnya n_estimator dan max_depth dan learning rate yang cukup tinggi melebihi standar.

In [115]:
Pipe = Pipeline([('CT', CT), ('XGB', XGBClassifier(n_estimators=200, learning_rate=0.04, max_depth=5, subsample=0.6))])
model = Pipe.fit(X_train, y_train)
accuracy_score(y_train, model.predict(X_train)), accuracy_score(y_test, model.predict(X_test)), accuracy_score(y_train, model.predict(X_train)) - accuracy_score(y_test, model.predict(X_test))

(0.928, 0.876, 0.052000000000000046)

Berdasarkan pemilihan yang saya telah tentukan. Kombinasi ini menghasilkan residual paling kecil sekaligus menghasilkan skor paling tinggi pada akurasi testing. Hal yang paling membantu adalah mengurangi learning rate dan max depth secara signifikan dan mengadakan subsample (menurun ketika subsample 0.7 atau 0.8).